In [21]:
import pandas as pd
import numpy as np
import os

print("Libraries loaded.")

Libraries loaded.


In [22]:
te_path = r"..\te_equivalence\NOTEBOOKS\outputs\PBM_Optimized_Candidates.csv"

te_data = pd.read_csv(te_path)

print("TE data loaded successfully.")
print("Shape:", te_data.shape)

display(
    te_data[
        [
            "Trade_Name",
            "Ingredient",
            "Dosage_Form",
            "Route_Of_Administration",
            "TE_Code"
        ]
    ].head(10)
)

TE data loaded successfully.
Shape: (10969, 37)


,Trade_Name,Ingredient,Dosage_Form,Route_Of_Administration,TE_Code
0,BUDESONIDE,BUDESONIDE,"AEROSOL, FOAM",RECTAL,AB
1,DUTASTERIDE AND TAMSULOSIN HYDROCHLORIDE,DUTASTERIDE; TAMSULOSIN HYDROCHLORIDE,CAPSULE,ORAL,AB
2,LISINOPRIL AND HYDROCHLOROTHIAZIDE,HYDROCHLOROTHIAZIDE; LISINOPRIL,TABLET,ORAL,AB
3,DOXYCYCLINE HYCLATE,DOXYCYCLINE HYCLATE,"TABLET, DELAYED RELEASE",ORAL,AB
4,PHENOXYBENZAMINE HYDROCHLORIDE,PHENOXYBENZAMINE HYDROCHLORIDE,CAPSULE,ORAL,AB
5,LINEZOLID,LINEZOLID,TABLET,ORAL,AB
6,LEVOCETIRIZINE DIHYDROCHLORIDE,LEVOCETIRIZINE DIHYDROCHLORIDE,TABLET,ORAL,AB
7,DOXERCALCIFEROL,DOXERCALCIFEROL,CAPSULE,ORAL,AB
8,BROMOCRIPTINE MESYLATE,BROMOCRIPTINE MESYLATE,TABLET,ORAL,AB
9,SODIUM PHENYLBUTYRATE,SODIUM PHENYLBUTYRATE,TABLET,ORAL,AB


In [23]:
# ============================================================
# CLEAN TE DATA
# ============================================================

te_rules = te_data.copy()

# Clean text columns
for col in [
    "Trade_Name",
    "Ingredient",
    "Dosage_Form",
    "Route_Of_Administration",
    "TE_Code"
]:
    
    te_rules[col] = (
        te_rules[col]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
    )

print("Data cleaned successfully.")

Data cleaned successfully.


In [24]:
# ============================================================
# CREATE FAST ALTERNATIVE LOOKUP
# ============================================================

alternative_lookup = {}

for _, row in te_rules.iterrows():

    key = (
        row["Ingredient"],
        row["Dosage_Form"],
        row["Route_Of_Administration"]
    )

    if key not in alternative_lookup:
        alternative_lookup[key] = []

    alternative_lookup[key].append(
        (
            row["Trade_Name"],
            row["TE_Code"]
        )
    )

print("Lookup created successfully.")
print("Number of groups:", len(alternative_lookup))

Lookup created successfully.
Number of groups: 709


In [25]:
# ============================================================
# FIND RULE-BASED TE ALTERNATIVE
# ============================================================

def find_alternative(row):

    key = (
        row["Ingredient"],
        row["Dosage_Form"],
        row["Route_Of_Administration"]
    )

    options = alternative_lookup.get(key, [])

    current_trade = row["Trade_Name"]

    # Find another product with the same
    # ingredient + dosage form + route
    for trade_name, te_code in options:

        if trade_name != current_trade:

            return pd.Series([
                trade_name,
                te_code,
                "TE-EQUIVALENT ALTERNATIVE"
            ])

    return pd.Series([
        np.nan,
        np.nan,
        "NO ALTERNATIVE FOUND"
    ])


# Apply the rule
te_rules[
    [
        "Alternative_Trade_Name",
        "Alternative_TE_Code",
        "TE_Decision"
    ]
] = te_rules.apply(
    find_alternative,
    axis=1
)

print("Alternative search completed.")

Alternative search completed.


In [26]:
# ============================================================
# CHECK RESULTS
# ============================================================

print("TE Decision distribution:")

print(
    te_rules["TE_Decision"].value_counts()
)

TE Decision distribution:
TE_Decision
NO ALTERNATIVE FOUND         9758
TE-EQUIVALENT ALTERNATIVE    1211
Name: count, dtype: int64


In [27]:
# ============================================================
# DISPLAY ALTERNATIVES
# ============================================================

alternatives = te_rules[
    te_rules["TE_Decision"] == "TE-EQUIVALENT ALTERNATIVE"
]

print("Number of alternatives found:", len(alternatives))

display(
    alternatives[
        [
            "Trade_Name",
            "Ingredient",
            "TE_Code",
            "Alternative_Trade_Name",
            "Alternative_TE_Code",
            "TE_Decision"
        ]
    ].head(30)
)

Number of alternatives found: 1211


,Trade_Name,Ingredient,TE_Code,Alternative_Trade_Name,Alternative_TE_Code,TE_Decision
26,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE,AB,TE-EQUIVALENT ALTERNATIVE
29,ALLOPURINOL,ALLOPURINOL,AB,LOPURIN,AB,TE-EQUIVALENT ALTERNATIVE
42,WARFARIN SODIUM,WARFARIN SODIUM,AB,JANTOVEN,AB,TE-EQUIVALENT ALTERNATIVE
51,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE,AB,TE-EQUIVALENT ALTERNATIVE
65,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,DOXORUBICIN HYDROCHLORIDE,AB,TE-EQUIVALENT ALTERNATIVE
80,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,TE-EQUIVALENT ALTERNATIVE
81,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,TE-EQUIVALENT ALTERNATIVE
91,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,DOXORUBICIN HYDROCHLORIDE,AB,TE-EQUIVALENT ALTERNATIVE
93,METHENAMINE HIPPURATE,METHENAMINE HIPPURATE,AB,UREX,AB,TE-EQUIVALENT ALTERNATIVE
96,"GRISEOFULVIN, ULTRAMICROSIZE","GRISEOFULVIN, ULTRAMICROSIZE",AB,FULVICIN P/G,AB,TE-EQUIVALENT ALTERNATIVE


In [28]:
# ============================================================
# SAVE RULE-BASED TE RESULTS
# ============================================================

output_dir = r"..\te_equivalence\NOTEBOOKS\outputs"

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "Rule_Based_TE_Alternatives.csv"
)

te_rules.to_csv(
    output_path,
    index=False
)

print("Saved successfully!")
print(output_path)

Saved successfully!
..\te_equivalence\NOTEBOOKS\outputs\Rule_Based_TE_Alternatives.csv


In [29]:
# ============================================================
# VIEW TE ALTERNATIVES
# ============================================================

alternatives = te_rules[
    te_rules["TE_Decision"] == "TE-EQUIVALENT ALTERNATIVE"
].copy()

print("Total TE alternatives:", len(alternatives))

display(
    alternatives[
        [
            "Trade_Name",
            "Ingredient",
            "TE_Code",
            "Alternative_Trade_Name",
            "Alternative_TE_Code",
            "TE_Decision"
        ]
    ].head(30)
)

Total TE alternatives: 1211


,Trade_Name,Ingredient,TE_Code,Alternative_Trade_Name,Alternative_TE_Code,TE_Decision
26,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE,AB,TE-EQUIVALENT ALTERNATIVE
29,ALLOPURINOL,ALLOPURINOL,AB,LOPURIN,AB,TE-EQUIVALENT ALTERNATIVE
42,WARFARIN SODIUM,WARFARIN SODIUM,AB,JANTOVEN,AB,TE-EQUIVALENT ALTERNATIVE
51,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE,AB,TE-EQUIVALENT ALTERNATIVE
65,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,DOXORUBICIN HYDROCHLORIDE,AB,TE-EQUIVALENT ALTERNATIVE
80,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,TE-EQUIVALENT ALTERNATIVE
81,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,TE-EQUIVALENT ALTERNATIVE
91,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,DOXORUBICIN HYDROCHLORIDE,AB,TE-EQUIVALENT ALTERNATIVE
93,METHENAMINE HIPPURATE,METHENAMINE HIPPURATE,AB,UREX,AB,TE-EQUIVALENT ALTERNATIVE
96,"GRISEOFULVIN, ULTRAMICROSIZE","GRISEOFULVIN, ULTRAMICROSIZE",AB,FULVICIN P/G,AB,TE-EQUIVALENT ALTERNATIVE


In [30]:
# ============================================================
# UNIQUE TE ALTERNATIVES
# ============================================================

print(
    "Unique original drugs:",
    alternatives["Trade_Name"].nunique()
)

print(
    "Unique alternative drugs:",
    alternatives["Alternative_Trade_Name"].nunique()
)

display(
    alternatives[
        [
            "Trade_Name",
            "Alternative_Trade_Name",
            "TE_Code",
            "Alternative_TE_Code"
        ]
    ].drop_duplicates().head(30)
)

Unique original drugs: 113
Unique alternative drugs: 82


,Trade_Name,Alternative_Trade_Name,TE_Code,Alternative_TE_Code
26,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,AB
29,ALLOPURINOL,LOPURIN,AB,AB
42,WARFARIN SODIUM,JANTOVEN,AB,AB
65,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,AB
80,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,AB
93,METHENAMINE HIPPURATE,UREX,AB,AB
96,"GRISEOFULVIN, ULTRAMICROSIZE",FULVICIN P/G,AB,AB
109,DILTIAZEM HYDROCHLORIDE,CARTIA XT,AB1,AB3
111,LEVOTHYROXINE SODIUM,LEVOLET,"AB1,AB2,AB3,AB4","AB1,AB2,AB3,AB4"
116,CLARAVIS,ZENATANE,AB1,AB1


In [31]:
# ============================================================
# CHECK TE ALTERNATIVES
# ============================================================

print("Total TE alternatives:", len(alternatives))

display(
    alternatives[
        [
            "Trade_Name",
            "Ingredient",
            "TE_Code",
            "Alternative_Trade_Name",
            "Alternative_TE_Code",
            "TE_Decision"
        ]
    ].drop_duplicates().head(30)
)

Total TE alternatives: 1211


,Trade_Name,Ingredient,TE_Code,Alternative_Trade_Name,Alternative_TE_Code,TE_Decision
26,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE,AB,TE-EQUIVALENT ALTERNATIVE
29,ALLOPURINOL,ALLOPURINOL,AB,LOPURIN,AB,TE-EQUIVALENT ALTERNATIVE
42,WARFARIN SODIUM,WARFARIN SODIUM,AB,JANTOVEN,AB,TE-EQUIVALENT ALTERNATIVE
65,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,DOXORUBICIN HYDROCHLORIDE,AB,TE-EQUIVALENT ALTERNATIVE
80,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,TE-EQUIVALENT ALTERNATIVE
93,METHENAMINE HIPPURATE,METHENAMINE HIPPURATE,AB,UREX,AB,TE-EQUIVALENT ALTERNATIVE
96,"GRISEOFULVIN, ULTRAMICROSIZE","GRISEOFULVIN, ULTRAMICROSIZE",AB,FULVICIN P/G,AB,TE-EQUIVALENT ALTERNATIVE
109,DILTIAZEM HYDROCHLORIDE,DILTIAZEM HYDROCHLORIDE,AB1,CARTIA XT,AB3,TE-EQUIVALENT ALTERNATIVE
111,LEVOTHYROXINE SODIUM,LEVOTHYROXINE SODIUM,"AB1,AB2,AB3,AB4",LEVOLET,"AB1,AB2,AB3,AB4",TE-EQUIVALENT ALTERNATIVE
116,CLARAVIS,ISOTRETINOIN,AB1,ZENATANE,AB1,TE-EQUIVALENT ALTERNATIVE


In [32]:
# ============================================================
# REMOVE DUPLICATE ALTERNATIVES
# ============================================================

alternatives_unique = alternatives[
    [
        "Trade_Name",
        "Ingredient",
        "TE_Code",
        "Alternative_Trade_Name",
        "Alternative_TE_Code",
        "TE_Decision"
    ]
].drop_duplicates().copy()

print("Unique TE alternative pairs:", len(alternatives_unique))

display(alternatives_unique.head(20))

Unique TE alternative pairs: 132


,Trade_Name,Ingredient,TE_Code,Alternative_Trade_Name,Alternative_TE_Code,TE_Decision
26,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE,AB,TE-EQUIVALENT ALTERNATIVE
29,ALLOPURINOL,ALLOPURINOL,AB,LOPURIN,AB,TE-EQUIVALENT ALTERNATIVE
42,WARFARIN SODIUM,WARFARIN SODIUM,AB,JANTOVEN,AB,TE-EQUIVALENT ALTERNATIVE
65,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,DOXORUBICIN HYDROCHLORIDE,AB,TE-EQUIVALENT ALTERNATIVE
80,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,TE-EQUIVALENT ALTERNATIVE
93,METHENAMINE HIPPURATE,METHENAMINE HIPPURATE,AB,UREX,AB,TE-EQUIVALENT ALTERNATIVE
96,"GRISEOFULVIN, ULTRAMICROSIZE","GRISEOFULVIN, ULTRAMICROSIZE",AB,FULVICIN P/G,AB,TE-EQUIVALENT ALTERNATIVE
109,DILTIAZEM HYDROCHLORIDE,DILTIAZEM HYDROCHLORIDE,AB1,CARTIA XT,AB3,TE-EQUIVALENT ALTERNATIVE
111,LEVOTHYROXINE SODIUM,LEVOTHYROXINE SODIUM,"AB1,AB2,AB3,AB4",LEVOLET,"AB1,AB2,AB3,AB4",TE-EQUIVALENT ALTERNATIVE
116,CLARAVIS,ISOTRETINOIN,AB1,ZENATANE,AB1,TE-EQUIVALENT ALTERNATIVE


In [33]:
# ============================================================
# EXAMPLES OF TE-EQUIVALENT ALTERNATIVES
# ============================================================

print("Sample TE-equivalent alternatives:")

display(
    alternatives_unique[
        alternatives_unique["TE_Decision"] == "TE-EQUIVALENT ALTERNATIVE"
    ].head(30)
)

Sample TE-equivalent alternatives:


,Trade_Name,Ingredient,TE_Code,Alternative_Trade_Name,Alternative_TE_Code,TE_Decision
26,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE,AB,TE-EQUIVALENT ALTERNATIVE
29,ALLOPURINOL,ALLOPURINOL,AB,LOPURIN,AB,TE-EQUIVALENT ALTERNATIVE
42,WARFARIN SODIUM,WARFARIN SODIUM,AB,JANTOVEN,AB,TE-EQUIVALENT ALTERNATIVE
65,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,DOXORUBICIN HYDROCHLORIDE,AB,TE-EQUIVALENT ALTERNATIVE
80,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE,AB,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,TE-EQUIVALENT ALTERNATIVE
93,METHENAMINE HIPPURATE,METHENAMINE HIPPURATE,AB,UREX,AB,TE-EQUIVALENT ALTERNATIVE
96,"GRISEOFULVIN, ULTRAMICROSIZE","GRISEOFULVIN, ULTRAMICROSIZE",AB,FULVICIN P/G,AB,TE-EQUIVALENT ALTERNATIVE
109,DILTIAZEM HYDROCHLORIDE,DILTIAZEM HYDROCHLORIDE,AB1,CARTIA XT,AB3,TE-EQUIVALENT ALTERNATIVE
111,LEVOTHYROXINE SODIUM,LEVOTHYROXINE SODIUM,"AB1,AB2,AB3,AB4",LEVOLET,"AB1,AB2,AB3,AB4",TE-EQUIVALENT ALTERNATIVE
116,CLARAVIS,ISOTRETINOIN,AB1,ZENATANE,AB1,TE-EQUIVALENT ALTERNATIVE


In [34]:
# ============================================================
# SAVE TE ALTERNATIVES
# ============================================================

output_dir = r"G:\PMS_Optimization\te_equivalence\outputs"

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "TE_Equivalent_Alternatives.csv"
)

alternatives_unique.to_csv(
    output_path,
    index=False
)

print("TE alternatives saved successfully.")
print(output_path)

TE alternatives saved successfully.
G:\PMS_Optimization\te_equivalence\outputs\TE_Equivalent_Alternatives.csv


In [35]:
# ============================================================
# PREPARE TE ALTERNATIVES FOR FORMULARY ANALYSIS
# ============================================================

formulary_te = alternatives_unique.copy()

formulary_te["Original_Drug"] = (
    formulary_te["Trade_Name"]
    .astype(str)
    .str.upper()
    .str.strip()
)

formulary_te["Alternative_Drug"] = (
    formulary_te["Alternative_Trade_Name"]
    .astype(str)
    .str.upper()
    .str.strip()
)

print("Formulary TE pairs:", len(formulary_te))

display(
    formulary_te[
        [
            "Original_Drug",
            "Alternative_Drug",
            "TE_Code",
            "Alternative_TE_Code"
        ]
    ].head(20)
)

Formulary TE pairs: 132


,Original_Drug,Alternative_Drug,TE_Code,Alternative_TE_Code
26,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,AB
29,ALLOPURINOL,LOPURIN,AB,AB
42,WARFARIN SODIUM,JANTOVEN,AB,AB
65,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,AB
80,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,AB
93,METHENAMINE HIPPURATE,UREX,AB,AB
96,"GRISEOFULVIN, ULTRAMICROSIZE",FULVICIN P/G,AB,AB
109,DILTIAZEM HYDROCHLORIDE,CARTIA XT,AB1,AB3
111,LEVOTHYROXINE SODIUM,LEVOLET,"AB1,AB2,AB3,AB4","AB1,AB2,AB3,AB4"
116,CLARAVIS,ZENATANE,AB1,AB1


In [36]:
# ============================================================
# CLEAN UNIQUE ALTERNATIVES
# ============================================================

alternative_pairs = alternatives[
    [
        "Trade_Name",
        "Alternative_Trade_Name",
        "TE_Code",
        "Alternative_TE_Code"
    ]
].drop_duplicates().copy()

# Remove cases where original and alternative are the same
alternative_pairs = alternative_pairs[
    alternative_pairs["Trade_Name"] !=
    alternative_pairs["Alternative_Trade_Name"]
].copy()

print("Unique alternative pairs:", len(alternative_pairs))

display(alternative_pairs.head(30))

Unique alternative pairs: 132


,Trade_Name,Alternative_Trade_Name,TE_Code,Alternative_TE_Code
26,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),DESMOPRESSIN ACETATE,AB,AB
29,ALLOPURINOL,LOPURIN,AB,AB
42,WARFARIN SODIUM,JANTOVEN,AB,AB
65,DOXORUBICIN HYDROCHLORIDE (LIPOSOMAL),DOXORUBICIN HYDROCHLORIDE,AB,AB
80,DESMOPRESSIN ACETATE,DESMOPRESSIN ACETATE (NEEDS NO REFRIGERATION),AB,AB
93,METHENAMINE HIPPURATE,UREX,AB,AB
96,"GRISEOFULVIN, ULTRAMICROSIZE",FULVICIN P/G,AB,AB
109,DILTIAZEM HYDROCHLORIDE,CARTIA XT,AB1,AB3
111,LEVOTHYROXINE SODIUM,LEVOLET,"AB1,AB2,AB3,AB4","AB1,AB2,AB3,AB4"
116,CLARAVIS,ZENATANE,AB1,AB1


In [37]:
# ============================================================
# SAVE TE ALTERNATIVES
# ============================================================

output_path = r"G:\PMS_Optimization\formulary_impact\outputs\TE_Rule_Based_Alternatives.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

alternative_pairs.to_csv(
    output_path,
    index=False
)

print("TE alternatives saved successfully.")
print(output_path)

TE alternatives saved successfully.
G:\PMS_Optimization\formulary_impact\outputs\TE_Rule_Based_Alternatives.csv


In [38]:
print("UTILIZATION COLUMNS:")
for col in utilization.columns:
    print(col)

UTILIZATION COLUMNS:


NameError: name 'utilization' is not defined